# Feature Recovery: ResidMLP + SPD Pipeline

This notebook trains a dictionary-initialized ResidMLP target, runs SPD on it, and analyzes the recovered component directions.

In [ ]:
from pathlib import Path
import json
import subprocess

REPO_ROOT = Path.cwd()
TARGET_CONFIG = REPO_ROOT / 'koko_experiments/feature_recovery/configs/resid_mlp_dictionary.yaml'
SPD_CONFIG = REPO_ROOT / 'koko_experiments/feature_recovery/configs/spd_resid_mlp_dictionary.yaml'
TRAIN_SCRIPT = REPO_ROOT / 'koko_experiments/feature_recovery/train_resid_mlp_target.py'
SPD_SCRIPT = REPO_ROOT / 'koko_experiments/feature_recovery/run_resid_mlp_spd.py'
ANALYZE_SCRIPT = REPO_ROOT / 'koko_experiments/feature_recovery/analyze_spd_run.py'


## Train the target model

In [ ]:
train_result = subprocess.run(['python', str(TRAIN_SCRIPT), str(TARGET_CONFIG)], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
print(train_result.stdout)
target_dir = Path(train_result.stdout.strip().splitlines()[-1])
target_dir

## Run SPD on the trained target

In [ ]:
spd_result = subprocess.run(['python', str(SPD_SCRIPT), str(SPD_CONFIG), str(target_dir)], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
print(spd_result.stdout)
spd_dir = Path(spd_result.stdout.strip().splitlines()[-1])
spd_dir

## Analyze recovered directions against the true dictionary

In [ ]:
analysis_result = subprocess.run(['python', str(ANALYZE_SCRIPT), str(spd_dir), str(target_dir)], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
print(analysis_result.stdout)
analysis_path = Path(analysis_result.stdout.strip().splitlines()[-1])
analysis = json.loads(analysis_path.read_text())
analysis

## Inspect the raw SPD metrics log

In [ ]:
metrics_path = spd_dir / 'metrics.jsonl'
lines = metrics_path.read_text().strip().splitlines()
json.loads(lines[-1]) if lines else {}